# Day 46 — Backpropagation and a multilayer perceptron in pure NumPy

Day 44 built a scalar autograd whose nodes were single numbers. Day 45 replaced the
update rule at the end of the training loop. This notebook puts a real model in the
middle: a two-layer network, trained on a three-armed spiral that no straight line can
separate, with every gradient worked out by hand on whole arrays.

Backpropagation through a linear layer is three lines, and each one is a transpose away
from the forward pass:

```python
dW = x.T @ dz
db = dz.sum(axis=0)
dx = dz @ W.T
```

The classification losses come with it, because an MLP's output layer needs them: softmax,
cross entropy, and the one identity that makes them worth writing as a single function -
their composed gradient is `(p - y) / n`.

Most of the day is about the ways these fail without raising: softmax on an axis that is
not the batch axis still returns numbers in [0, 1]; a cross entropy that forms
probabilities first takes `log(0)`; `db = dz.sum()` broadcasts happily; `dW = x @ dz`
runs whenever the batch size equals the input width; and all-zero initialisation leaves
64 hidden units behaving as one forever.

In [1]:
import math

import numpy as np

np.seterr(all='ignore')      # the deliberately unstable versions overflow on purpose

## 1. softmax and cross entropy, written so they cannot overflow

Subtracting the row maximum changes nothing mathematically: `exp(z - m)` and `exp(z)`
differ by a constant that cancels in the ratio. Numerically it is the difference between
an answer and a `nan`, because `exp` never sees a positive argument.

`log_softmax` goes further and never forms the probability at all, so the log is taken of
an expression rather than of a number that may have underflowed to zero.

In [2]:
# ================================================================ 1. softmax, safely
def softmax(z, axis=-1):
    """Subtracting the row max changes nothing mathematically and everything
    numerically: exp() never sees a positive argument, so it cannot overflow."""
    z = z - np.max(z, axis=axis, keepdims=True)
    e = np.exp(z)
    return e / np.sum(e, axis=axis, keepdims=True)


def softmax_naive(z, axis=-1):
    e = np.exp(z)
    return e / np.sum(e, axis=axis, keepdims=True)


def log_softmax(z, axis=-1):
    """log(softmax(z)) without ever forming softmax(z): log-sum-exp."""
    m = np.max(z, axis=axis, keepdims=True)
    return z - m - np.log(np.sum(np.exp(z - m), axis=axis, keepdims=True))


def cross_entropy(logits, y):
    """Mean cross entropy of integer labels y given raw logits."""
    n = len(y)
    return float(-np.mean(log_softmax(logits)[np.arange(n), y]))


def cross_entropy_naive(logits, y):
    """The textbook spelling: form the probabilities, then take their log."""
    n = len(y)
    p = softmax_naive(logits)
    return float(-np.mean(np.log(p[np.arange(n), y])))


big = np.array([[1000.0, 1001.0, 1002.0]])
print('naive ', softmax_naive(big)[0])
print('stable', np.round(softmax(big)[0], 6))
lg = np.array([[-400.0, 0.0, 400.0]])
print('cross entropy, true class 0: stable', cross_entropy(lg, np.array([0])),
      ' naive', cross_entropy_naive(lg, np.array([0])))

naive  [nan nan nan]
stable [0.090031 0.244728 0.665241]
cross entropy, true class 0: stable 800.0  naive inf


## 2. The axis bug that still returns probabilities

`softmax(z, axis=0)` normalises down the batch instead of across the classes. Every entry
is still between 0 and 1 and something still sums to 1, so nothing complains - the model
is just being told that classes compete across examples rather than within one.

In [3]:
z = np.array([[2.0, 1.0, 0.1], [0.5, 2.5, 0.2]])
right, wrong = softmax(z, axis=1), softmax(z, axis=0)
print('axis=1 row sums', right.sum(axis=1), ' <- what we want')
print('axis=0 row sums', wrong.sum(axis=1), ' column sums', wrong.sum(axis=0))
print('argmax per row:', right.argmax(1), 'vs', wrong.argmax(1),
      '- the loss goes wrong before the prediction does')

axis=1 row sums [1. 1.]  <- what we want
axis=0 row sums [1.47502081 1.52497919]  column sums [1. 1. 1.]
argmax per row: [0 1] vs [0 1] - the loss goes wrong before the prediction does


## 3. sigmoid and binary cross entropy

The same story for two classes. `1 / (1 + exp(-x))` overflows for large negative `x`;
branching on the sign keeps `exp` on the safe side. The value it returns is right anyway -
it is `log(sigmoid(x))` that turns into `-inf`, and the loss is the thing being
differentiated.

In [4]:
# ================================================================ 2. sigmoid, safely
def sigmoid(x):
    """exp() only ever sees a negative argument, on both branches."""
    out = np.empty_like(np.asarray(x, dtype=float))
    pos = np.asarray(x) >= 0
    out[pos] = 1.0 / (1.0 + np.exp(-np.asarray(x)[pos]))
    ex = np.exp(np.asarray(x)[~pos])
    out[~pos] = ex / (1.0 + ex)
    return out


def sigmoid_naive(x):
    return 1.0 / (1.0 + np.exp(-np.asarray(x, dtype=float)))


def log_sigmoid(x):
    """log(sigmoid(x)) = -log1p(exp(-x)), written so exp() never overflows."""
    x = np.asarray(x, dtype=float)
    return np.where(x >= 0, -np.log1p(np.exp(-np.abs(x))),
                    x - np.log1p(np.exp(-np.abs(x))))


def log_sigmoid_naive(x):
    return np.log(sigmoid_naive(x))


def bce_with_logits(z, y):
    """Binary cross entropy straight from the logits, never forming log(p)."""
    return float(np.mean(np.logaddexp(0.0, z) - y * z))


xs = np.array([-800.0, -40.0, 0.0, 40.0, 800.0])
print('sigmoid     ', sigmoid(xs))
print('log_sigmoid ', np.round(log_sigmoid(xs), 4))
print('naive log   ', log_sigmoid_naive(xs))
print('BCE from logits at z = -800, label 1:', bce_with_logits(np.array([-800.0]), np.array([1.0])))

sigmoid      [0.00000000e+00 4.24835426e-18 5.00000000e-01 1.00000000e+00
 1.00000000e+00]
log_sigmoid  [-8.000e+02 -4.000e+01 -6.931e-01 -0.000e+00 -0.000e+00]
naive log    [        -inf -40.          -0.69314718   0.           0.        ]
BCE from logits at z = -800, label 1: 800.0


## 4. The gradient of softmax + cross entropy is `p - y`

Differentiating the two separately means a Jacobian for the softmax and a division by `p`
for the log. Composed, they cancel: everything collapses to the predicted distribution
minus the one-hot label. This is the only reason the two are implemented as one function.

In [5]:
def softmax_ce_grad(logits, y, mean=True):
    """The whole reason softmax and cross entropy are always written together:
    their composed derivative collapses to (p - y), and dividing by n makes it the
    gradient of the *mean* loss - which is the loss we actually reported."""
    n = len(y)
    p = softmax(logits)
    p[np.arange(n), y] -= 1.0
    return p / n if mean else p


# ================================================================ 4. gradient checking
def numeric_grad(f, a, h=1e-5):
    """Central differences over every entry of the array a, in place."""
    g = np.zeros_like(a)
    it = np.nditer(a, flags=['multi_index'])
    while not it.finished:
        i = it.multi_index
        old = a[i]
        a[i] = old + h
        up = f()
        a[i] = old - h
        dn = f()
        a[i] = old
        g[i] = (up - dn) / (2 * h)
        it.iternext()
    return g


rng = np.random.default_rng(46)
logits = rng.normal(size=(5, 4))
y = rng.integers(0, 4, size=5)
g = softmax_ce_grad(logits.copy(), y)
num = numeric_grad(lambda: cross_entropy(logits, y), logits)
print('on the logits drawn in this cell, analytic vs central differences:', np.max(np.abs(g - num)))
print('(mlp.py reaches this check with its generator further along, so it draws')
print(' different logits and reports a different - equally tiny - difference)')

on the logits drawn in this cell, analytic vs central differences: 1.9859960898038764e-11
(mlp.py reaches this check with its generator further along, so it draws
 different logits and reports a different - equally tiny - difference)


## 5. The layers

A `Linear` layer caches its input on the way forward and uses it on the way back. The
`bug` flags are there to make each failure reproducible later in the notebook rather than
described in prose.

In [6]:
# ================================================================ 3. the layers
class Linear:
    """z = x @ W + b.  Backward is three lines, and each one is a transpose away
    from the forward pass."""

    def __init__(self, d_in, d_out, rng, scale=None, zero=False):
        if zero:
            self.W = np.zeros((d_in, d_out))
        else:
            s = scale if scale is not None else math.sqrt(2.0 / d_in)   # He init
            self.W = rng.normal(scale=s, size=(d_in, d_out))
        self.b = np.zeros(d_out)
        self.x = None

    def forward(self, x):
        self.x = x
        return x @ self.W + self.b

    def backward(self, dz, bug=None):
        # dz already carries the 1/n from the mean loss, so there is no second divide
        if bug == 'no_transpose':
            self.dW = self.x @ dz                # only runs at all when n == d_in
        else:
            self.dW = self.x.T @ dz
        self.db = dz.sum() if bug == 'scalar_bias' else dz.sum(axis=0)
        return dz @ self.W.T

    @property
    def params(self):
        return [self.W, self.b]

    @property
    def grads(self):
        return [self.dW, self.db]


class ReLU:
    def forward(self, x):
        self.mask = x > 0
        return x * self.mask

    def backward(self, dz, bug=None):
        return dz * self.mask

    params = grads = ()


class Tanh:
    def forward(self, x):
        self.out = np.tanh(x)
        return self.out

    def backward(self, dz, bug=None):
        return dz * (1.0 - self.out ** 2)

    params = grads = ()


class MLP:
    """A list of layers, a forward pass that caches, and a backward pass that walks
    the same list in reverse - the reverse topological order of Day 13, except the
    graph is a straight line so reversing the list is enough."""

    def __init__(self, sizes, rng, act=ReLU, zero=False):
        self.layers = []
        for i in range(len(sizes) - 1):
            self.layers.append(Linear(sizes[i], sizes[i + 1], rng, zero=zero))
            if i < len(sizes) - 2:
                self.layers.append(act())

    def forward(self, x):
        for layer in self.layers:
            x = layer.forward(x)
        return x

    def backward(self, dlogits, bug=None):
        for layer in reversed(self.layers):
            dlogits = layer.backward(dlogits, bug)
        return dlogits

    def loss_and_grad(self, x, y, bug=None):
        logits = self.forward(x)
        loss = cross_entropy(logits, y)
        self.backward(softmax_ce_grad(logits, y, mean=(bug != 'no_mean')), bug)
        return loss

    @property
    def params(self):
        return [p for l in self.layers for p in l.params]

    @property
    def grads(self):
        return [g for l in self.layers for g in l.grads]


net = MLP([2, 64, 3], np.random.default_rng(0))
print(' -> '.join(type(l).__name__ for l in net.layers))
print('parameter shapes:', [tuple(p.shape) for p in net.params])

Linear -> ReLU -> Linear
parameter shapes: [(2, 64), (64,), (64, 3), (3,)]


## 6. Gradient checking

Day 44's central differences, applied to every entry of every parameter array. This is
the test that catches all four bugs below, and it is worth running once on any gradient
written by hand.

In [7]:
def gradcheck(net, x, y, bug=None, h=1e-5):
    """Largest relative error between backprop and central differences."""
    net.loss_and_grad(x, y, bug)
    analytic = [g.copy() for g in net.grads]
    worst = 0.0
    for p, a in zip(net.params, analytic):
        num = numeric_grad(lambda: cross_entropy(net.forward(x), y), p, h)
        denom = np.maximum(np.abs(a) + np.abs(num), 1e-12)
        worst = max(worst, float(np.max(np.abs(a - num) / denom)))
    return worst


# ================================================================ 5. data and training
def spiral(points=100, classes=3, seed=46, noise=0.22):
    """The classic spiral: three arms that no straight line can separate."""
    rng = np.random.default_rng(seed)
    X = np.zeros((points * classes, 2))
    y = np.zeros(points * classes, dtype=int)
    for c in range(classes):
        idx = range(points * c, points * (c + 1))
        r = np.linspace(0.05, 1.0, points)   # not 0: all three arms would share the origin
        t = np.linspace(c * 4, (c + 1) * 4, points) + rng.normal(scale=noise, size=points)
        X[idx] = np.c_[r * np.sin(t), r * np.cos(t)]
        y[idx] = c
    return X, y


X, y = spiral()
print('spiral:', X.shape, 'points,', y.max() + 1, 'classes')
print('gradcheck, correct backward:', gradcheck(MLP([2, 8, 3], np.random.default_rng(1)), X[:12], y[:12]))

spiral: (300, 2) points, 3 classes
gradcheck, correct backward: 7.849050936641177e-09


## 7. Training it

Adam from Day 45, 1,200 full-batch steps. The comparison that matters is the second
model: with no hidden layer this is softmax regression, and three straight boundaries
cannot follow three spiral arms.

In [8]:
class Adam:
    """Day 45's optimiser, the version with bias correction and eps outside the sqrt."""

    def __init__(self, params, lr=0.05, b1=0.9, b2=0.999, eps=1e-8):
        self.params, self.lr, self.b1, self.b2, self.eps = params, lr, b1, b2, eps
        self.m = [np.zeros_like(p) for p in params]
        self.v = [np.zeros_like(p) for p in params]
        self.t = 0

    def step(self, grads):
        self.t += 1
        for p, g, m, v in zip(self.params, grads, self.m, self.v):
            m *= self.b1
            m += (1 - self.b1) * g
            v *= self.b2
            v += (1 - self.b2) * g * g
            mh = m / (1 - self.b1 ** self.t)
            vh = v / (1 - self.b2 ** self.t)
            p -= self.lr * mh / (np.sqrt(vh) + self.eps)


def accuracy(net, X, y):
    return float(np.mean(net.forward(X).argmax(axis=1) == y))


def train(net, X, y, steps=1200, lr=0.05, bug=None, every=200):
    opt = Adam(net.params, lr=lr)
    hist = []
    for t in range(1, steps + 1):
        loss = net.loss_and_grad(X, y, bug)
        opt.step(net.grads)
        if t % every == 0 or t == 1:
            hist.append((t, loss, accuracy(net, X, y)))
    return hist


def hidden_activations(net, X):
    """What the hidden layer sees, to show zero init collapsing every unit."""
    h = net.layers[0].forward(X)
    return net.layers[1].forward(h)


net = MLP([2, 64, 3], np.random.default_rng(0))
for t, loss, acc in train(net, X, y):
    print(f'step {t:>5}  loss {loss:.4f}  accuracy {acc:.3f}')
lin = MLP([2, 3], np.random.default_rng(0))
train(lin, X, y)
print('hidden layer:', accuracy(net, X, y), '  no hidden layer:', accuracy(lin, X, y))
print('(training accuracy, not a generalisation claim)')

step     1  loss 1.1068  accuracy 0.633
step   200  loss 0.0036  accuracy 1.000
step   400  loss 0.0011  accuracy 1.000
step   600  loss 0.0005  accuracy 1.000
step   800  loss 0.0003  accuracy 1.000
step  1000  loss 0.0002  accuracy 1.000
step  1200  loss 0.0001  accuracy 1.000
hidden layer: 1.0   no hidden layer: 0.53
(training accuracy, not a generalisation claim)


## 8. Four bugs that never raise

Each one is a single character or a single missing operation away from correct, and each
one leaves a training loop that runs to completion and produces a model.

In [9]:
for bug, label in (('scalar_bias', 'db = dz.sum() instead of sum(axis=0)'),
                   ('no_mean', 'gradient of the sum, loss of the mean')):
    print(f'{label:<40} gradcheck {gradcheck(MLP([2, 8, 3], np.random.default_rng(1)), X[:12], y[:12], bug=bug):.2e}')
n = 8
Xq, yq = rng.normal(size=(n, n)), rng.integers(0, 3, size=n)
print(f'{"dW = x @ dz, batch == input width":<40} gradcheck '
      f'{gradcheck(MLP([n, 8, 3], np.random.default_rng(1)), Xq, yq, bug="no_transpose"):.2e}')
try:
    MLP([2, 8, 3], np.random.default_rng(1)).loss_and_grad(X[:12], y[:12], bug='no_transpose')
except ValueError as e:
    print(f'{"the same bug, batch != input width":<40} {type(e).__name__}: shapes do not line up')
zero = MLP([2, 64, 3], np.random.default_rng(0), zero=True)
train(zero, X, y, steps=400)
h = hidden_activations(zero, X[:4])
print(f'{"all-zero init":<40} {len(np.unique(np.round(h, 9)))} distinct hidden values '
      f'out of {h.size}, accuracy {accuracy(zero, X, y):.3f}')

db = dz.sum() instead of sum(axis=0)     gradcheck 1.00e+00
gradient of the sum, loss of the mean    gradcheck 8.46e-01
dW = x @ dz, batch == input width        gradcheck 1.00e+00
the same bug, batch != input width       ValueError: shapes do not line up
all-zero init                            1 distinct hidden values out of 256, accuracy 0.333


## 9. One failure that is not a bug

ReLU has no derivative at zero. Backpropagation picks a side (`mask = x > 0`, so zero);
central differences average both sides and get 0.5. Feed the network a row of zeros and
the pre-activations land exactly on the corner, and a perfectly correct gradient fails its
own check. Gradient-check with a smooth activation instead.

In [10]:
def kink_demo(h=1e-5):
    """A gradient check that fails even though backprop is right.

    ReLU has no derivative at 0.  Back-propagation picks one side (mask = x > 0, so
    zero); central differences average both sides and get 0.5.  Feed the network a
    row of zeros and the biases start at zero, so the pre-activation is exactly 0 and
    the two disagree.  Nothing is wrong with the code - the function has a corner.
    """
    x = np.zeros((2, 3))
    x[1] = [0.4, -0.2, 0.1]
    y = np.array([0, 1])
    relu_err = gradcheck(MLP([3, 4, 2], np.random.default_rng(3)), x, y, h=h)
    tanh_err = gradcheck(MLP([3, 4, 2], np.random.default_rng(3), act=Tanh), x, y, h=h)
    net = MLP([3, 4, 2], np.random.default_rng(3))
    pre = net.layers[0].forward(x)
    return relu_err, tanh_err, int(np.sum(pre == 0.0)), pre.size


r_err, t_err, zeros, total = kink_demo()
print(f'{zeros} of {total} pre-activations are exactly 0')
print(f'ReLU gradcheck {r_err:.2f},  the same network with tanh {t_err:.1e}')

4 of 8 pre-activations are exactly 0
ReLU gradcheck 1.00,  the same network with tanh 3.4e-10


## 10. LeetCode 867 and LeetCode 48

The transpose is the operation every backward rule is built on, and rotating a square
matrix is a transpose followed by a row reversal. Both carry the lesson from Day 44 that
NumPy's transpose only swaps strides, while a list of lists has to move every element.

In [11]:
# ================================================================ 6. LeetCode 867
class Solution867:
    """Transpose a matrix: the operation every backward rule is built on."""

    def transpose(self, matrix: list[list[int]]) -> list[list[int]]:
        m, n = len(matrix), len(matrix[0])
        out = [[0] * m for _ in range(n)]
        for i in range(m):
            for j in range(n):
                out[j][i] = matrix[i][j]
        return out


# ================================================================ 7. LeetCode 48
class Solution48:
    """Rotate a square matrix 90 degrees clockwise, in place: transpose, then
    reverse each row."""

    def rotate(self, matrix: list[list[int]]) -> None:
        n = len(matrix)
        for i in range(n):
            for j in range(i + 1, n):            # j > i only, or it swaps back
                matrix[i][j], matrix[j][i] = matrix[j][i], matrix[i][j]
        for row in matrix:
            row.reverse()


def rotate_full_loop(matrix):
    """The same rotation with j over the whole row: every pair is swapped twice,
    so the matrix comes back unchanged - and nothing raises."""
    n = len(matrix)
    out = [row[:] for row in matrix]
    for i in range(n):
        for j in range(n):
            out[i][j], out[j][i] = out[j][i], out[i][j]
    for row in out:
        row.reverse()
    return out


m = [[1, 2, 3], [4, 5, 6]]
print(m, '->', Solution867().transpose(m))
a = np.array(m)
print('numpy strides', a.strides, '->', a.T.strides, ' shares memory', np.shares_memory(a, a.T))
sq = [[1, 2, 3], [4, 5, 6], [7, 8, 9]]
got = [r[:] for r in sq]
Solution48().rotate(got)
print('rotated (j > i):       ', got)
print('j over every column:   ', rotate_full_loop(sq), ' <- every pair swapped twice')

[[1, 2, 3], [4, 5, 6]] -> [[1, 4], [2, 5], [3, 6]]
numpy strides (24, 8) -> (8, 24)  shares memory True
rotated (j > i):        [[7, 4, 1], [8, 5, 2], [9, 6, 3]]
j over every column:    [[3, 2, 1], [6, 5, 4], [9, 8, 7]]  <- every pair swapped twice


## Tests

In [12]:
def check() -> None:
    np.seterr(all='ignore')
    rng = np.random.default_rng(46)
    # softmax stability
    big = np.array([[1000.0, 1001.0, 1002.0]])
    assert np.isnan(softmax_naive(big)).all() and np.allclose(softmax(big).sum(), 1.0)
    assert abs(softmax(big)[0, 2] - math.exp(0) / (math.exp(-2) + math.exp(-1) + 1)) < 1e-12
    z = rng.normal(size=(6, 4)) * 3
    assert np.max(np.abs(softmax(z) - softmax_naive(z))) < 1e-12
    # the wrong axis still produces something that sums to one, just not per row
    w = softmax(z, axis=0)
    assert np.allclose(w.sum(axis=0), 1.0) and not np.allclose(w.sum(axis=1), 1.0)
    assert ((w >= 0) & (w <= 1)).all()
    # cross entropy
    lg = np.array([[0.0, 0.0, 800.0]])
    assert cross_entropy(lg, np.array([2])) < 1e-12
    assert math.isinf(cross_entropy_naive(lg, np.array([0])))
    assert abs(cross_entropy(lg, np.array([0])) - 800.0) < 1e-6
    # sigmoid
    xs = np.array([-800.0, 0.0, 800.0])
    assert np.isinf(sigmoid_naive(xs)[0]) or sigmoid_naive(xs)[0] == 0.0
    assert abs(sigmoid(xs)[1] - 0.5) < 1e-15 and sigmoid(xs)[2] == 1.0 and sigmoid(xs)[0] == 0.0
    assert math.isinf(log_sigmoid_naive(np.array([-800.0]))[0])
    assert abs(log_sigmoid(np.array([-800.0]))[0] + 800.0) < 1e-6
    assert abs(bce_with_logits(np.array([-800.0]), np.array([1.0])) - 800.0) < 1e-6
    for x in (-30.0, -1.0, 0.0, 1.0, 30.0):
        assert abs(log_sigmoid(np.array([x]))[0] - math.log(1 / (1 + math.exp(-x)))) < 1e-9
    # (p - y) / n really is the gradient
    logits = rng.normal(size=(5, 4))
    y = rng.integers(0, 4, size=5)
    g = softmax_ce_grad(logits.copy(), y)
    assert np.max(np.abs(g - numeric_grad(lambda: cross_entropy(logits, y), logits))) < 1e-8
    # backprop agrees with central differences, and each bug does not
    X, yy = spiral()
    good = gradcheck(MLP([2, 8, 3], np.random.default_rng(1)), X[:12], yy[:12])
    assert good < 1e-6, good
    assert gradcheck(MLP([2, 8, 3], np.random.default_rng(1), act=Tanh), X[:12], yy[:12]) < 1e-6
    # the three arms must not share the origin, or one point carries three labels
    assert np.min(np.linalg.norm(X[:, None] - X[None, :], axis=-1)
                  + np.eye(len(X)) * 9) > 1e-6
    # a ReLU kink makes a correct gradient look wrong; tanh does not
    r_err, t_err, zeros, _ = kink_demo()
    assert zeros > 0 and r_err > 0.2 and t_err < 1e-6
    for bug in ('scalar_bias', 'no_mean'):
        assert gradcheck(MLP([2, 8, 3], np.random.default_rng(1)), X[:12], yy[:12], bug=bug) > 0.1
    n_eq = 8
    Xq, yq = rng.normal(size=(n_eq, n_eq)), rng.integers(0, 3, size=n_eq)
    net_q = MLP([n_eq, 8, 3], np.random.default_rng(1))
    net_q.loss_and_grad(Xq, yq, bug='no_transpose')          # runs only because n == d_in
    assert net_q.layers[0].dW.shape == (n_eq, 8)
    assert gradcheck(MLP([n_eq, 8, 3], np.random.default_rng(1)), Xq, yq, bug='no_transpose') > 0.1
    try:
        MLP([2, 8, 3], np.random.default_rng(1)).loss_and_grad(X[:12], yy[:12], bug='no_transpose')
        raise AssertionError('expected a shape error once n != d_in')
    except ValueError:
        pass
    # training
    net = MLP([2, 64, 3], np.random.default_rng(0))
    train(net, X, yy)
    assert accuracy(net, X, yy) > 0.95
    lin = MLP([2, 3], np.random.default_rng(0))
    train(lin, X, yy)
    assert accuracy(lin, X, yy) < 0.6
    # symmetry never breaks from an all-zero start
    zero = MLP([2, 64, 3], np.random.default_rng(0), zero=True)
    train(zero, X, yy, steps=400)
    h = hidden_activations(zero, X[:4])
    assert len(np.unique(np.round(h, 9))) <= 4 and accuracy(zero, X, yy) < 0.6
    # LeetCode
    assert Solution867().transpose([[1, 2, 3], [4, 5, 6]]) == [[1, 4], [2, 5], [3, 6]]
    assert Solution867().transpose([[1, 2], [3, 4]]) == [[1, 3], [2, 4]]
    for shape in ((2, 3), (3, 2), (1, 5), (4, 4)):
        a = rng.integers(0, 9, size=shape)
        assert Solution867().transpose(a.tolist()) == a.T.tolist()
    m = [[1, 2, 3], [4, 5, 6], [7, 8, 9]]
    got = [r[:] for r in m]
    Solution48().rotate(got)
    assert got == [[7, 4, 1], [8, 5, 2], [9, 6, 3]] == np.rot90(np.array(m), -1).tolist()
    assert rotate_full_loop(m) == [r[::-1] for r in m] != got
    for n in (1, 2, 3, 5):
        a = rng.integers(0, 9, size=(n, n))
        g2 = a.tolist()
        Solution48().rotate(g2)
        assert g2 == np.rot90(a, -1).tolist()


check()
print('all assertions passed')

all assertions passed
